In [1]:
!pip install -U ydata-profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.8 MB/s eta 0:00:00


In [15]:
#데이터 불러오기
import pandas as pd
from ydata_profiling import ProfileReport
data = pd.read_csv('/content/dataset.csv')


In [16]:
# 타깃 피처 만들기
df = data[data['isOpen'] == 0].copy() # isOpen == 1인 경우 아직 거래가 완전히 끝나지 않은 불완전한 데이터이기 때문에 clear_date 값이 존재하지 않음
df['clear_date'] = pd.to_datetime(df['clear_date'], errors = 'coerce')
df['due_in_date'] = pd.to_datetime(df['due_in_date'].astype(int).astype(str),format='%Y%m%d' ,errors = 'coerce')
df['target'] = (df['clear_date'] > df['due_in_date']).astype(int)


In [17]:
# 중복 행 제거
before_rows = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed duplicate rows: {before_rows - len(df):,}")
print(f"Remaining rows: {len(df):,}")

Removed duplicate rows: 842
Remaining rows: 39,158


In [18]:
# 학습 데이터, 테스트 데이터 나누기
df = df.sort_values('baseline_create_date').reset_index(drop=True) # 과거의 학습 데이터로 미래의 데이터를 분류해야하기 때문에 시간 순서로 정렬
split_idx = int(len(df) * 0.8) # train : test = 8 : 2로 분리
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

In [19]:
# 학습 데이터, 테스트 데이터 다운로드
train_df.to_csv("/content/train.csv", index=False)
test_df.to_csv("/content/test.csv", index=False)


In [14]:
# 기초 EDA 정보 다운로드
report = ProfileReport(train_df)
report.to_file("content/data_report.html")

/usr/local/lib/python3.12/dist-packages/ydata_profiling/utils/dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"index": "df_index"}, inplace=True)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 20/20 [00:01<00:00, 18.02it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

FileNotFoundError: [Errno 2] No such file or directory: 'content/data_report.html'